<a href="https://colab.research.google.com/github/SaraEstilaei/incident-sentiment-analyzer/blob/Model%2FNLP_02_DeBERTa/NLP_02_deberta_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn gdown sentencepiece

import os, gc, glob, zipfile, inspect
import numpy as np
import pandas as pd
import torch
import gdown
from google.colab import drive
from scipy.special import softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, log_loss, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

<div dir="rtl">

# مدل ۱: DeBERTa-v3-base روی 80K نقد آمازون

**هدف:** پیش‌بینی امتیاز ۱ تا ۵ از `full_text` (summary + reviewText)  
**معیار:** micro-F1 (اصلی) و macro-F1 (مکمل، چون داده نامتوازنه)  
**خروجی‌ها:** مدل، split های train/val/test و احتمال‌های val/test برای ensemble

</div>

In [ ]:
drive.mount("/content/drive")      # فقط برای ذخیره‌ی مدل، چک‌پوینت و خروجی‌ها

SEED       = 42
MODEL_NAME = "microsoft/deberta-v3-base"
RUN_NAME   = "deberta_80k"
MAX_LEN    = 128
EPOCHS     = 3
LR         = 2e-5
BS         = 32

N_SELECT = 80_000                  # تعداد نمونه‌ی انتخاب‌شده از فایل اصلی
N_TEST   = 8_000                   # 10%
N_VAL    = 16_000                  # 20%  (بقیه: 70% برای train)

SAVE_ROOT = "/content/drive/MyDrive/NLP_Proj/deberta_80k"
CKPT_DIR  = f"{SAVE_ROOT}/ckpt"
os.makedirs(SAVE_ROOT, exist_ok=True)

set_seed(SEED)
assert torch.cuda.is_available(), "GPU روشن نیست: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: Tesla T4


In [ ]:
ZIP_ID = "1Dzp8Ks1aQoscJA6XLRXg-OtqunshlMfJ"

if not os.path.exists("/content/raw/train"):
    os.makedirs("/content/raw", exist_ok=True)
    gdown.download(id=ZIP_ID, output="/content/raw/train.zip", quiet=False)
    with zipfile.ZipFile("/content/raw/train.zip") as z:
        print(z.namelist())
        z.extractall("/content/raw/train")

CSV_PATH = max(glob.glob("/content/raw/train/**/*.csv", recursive=True), key=os.path.getsize)
print("CSV:", CSV_PATH, round(os.path.getsize(CSV_PATH) / 1e6), "MB")

Downloading...
From (original): https://drive.google.com/uc?id=1Dzp8Ks1aQoscJA6XLRXg-OtqunshlMfJ
From (redirected): https://drive.google.com/uc?id=1Dzp8Ks1aQoscJA6XLRXg-OtqunshlMfJ&confirm=t&uuid=8349b788-71f8-42b3-867e-082e0f9aa367
To: /content/raw/train.zip
100%|██████████| 290M/290M [00:04<00:00, 63.0MB/s]


['df cleaned/', 'df cleaned/df_cleaned.csv']
CSV: /content/raw/train/df cleaned/df_cleaned.csv 1712 MB


In [ ]:
df = pd.read_csv(CSV_PATH, usecols=["overall", "vote", "full_text"])
print("کل ردیف‌ها:", df.shape)

df["overall"] = df["overall"].astype(int)
df["vote"]    = pd.to_numeric(df["vote"], errors="coerce").fillna(0).astype(int)
df = df.dropna(subset=["full_text"])

n0 = len(df)
df = df.sort_values("vote", ascending=False).drop_duplicates(subset=["full_text"]).reset_index(drop=True)
print(f"حذف تکراری‌ها: {n0 - len(df)} | باقی‌مانده: {len(df)}")

print("\nتوزیع امتیازها (فایل اصلی):")
print(df["overall"].value_counts(normalize=True).sort_index().round(4))
print("\nتوزیع vote:")
print(df["vote"].describe())

کل ردیف‌ها: (838944, 3)
حذف تکراری‌ها: 30468 | باقی‌مانده: 808476

توزیع امتیازها (فایل اصلی):
overall
1    0.0986
2    0.0678
3    0.0969
4    0.1871
5    0.5496
Name: proportion, dtype: float64

توزیع vote:
count    808476.000000
mean          1.747615
std          13.371707
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        2846.000000
Name: vote, dtype: float64


In [ ]:
# 1) سهم هر امتیاز از ۸۰ هزار (به نسبت توزیع فایل اصلی؛ بزرگ‌ترین باقی‌مانده برای رسیدن به مجموع دقیق)
share = df["overall"].value_counts(normalize=True).sort_index()
raw   = share * N_SELECT
n_k   = np.floor(raw).astype(int)
rem   = N_SELECT - n_k.sum()
n_k.loc[(raw - n_k).sort_values(ascending=False).index[:rem]] += 1
print("تعداد انتخاب از هر امتیاز:", n_k.to_dict(), "| مجموع:", n_k.sum())

# 2) داخل هر امتیاز: بیشترین vote اول؛ بین نمونه‌های هم‌vote در مرز انتخاب، تصادفی
rng = np.random.RandomState(SEED)
df["_r"] = rng.rand(len(df))

parts = []
for k, n in n_k.items():
    g = df[df["overall"] == k].sort_values(["vote", "_r"], ascending=[False, True])
    chosen = g.head(n)
    cutoff = chosen["vote"].min()
    print(f"امتیاز {k}: انتخاب {n} از {len(g)} | کمترین vote انتخاب‌شده: {cutoff} | "
          f"هم‌vote در مرز: {(g['vote'] == cutoff).sum()} (انتخاب {(chosen['vote'] == cutoff).sum()})")
    parts.append(chosen)

sel = pd.concat(parts).drop(columns="_r").reset_index(drop=True)
sel["label"] = sel["overall"] - 1                                 # 1..5 -> 0..4
del df, parts; gc.collect()

# 3) چک: توزیع برچسب‌ها عوض نشده باشد
print("\nتوزیع امتیازها بعد از انتخاب:")
print(sel["overall"].value_counts(normalize=True).sort_index().round(4))
print("\nvote در نمونه‌ی انتخاب‌شده:")
print(sel["vote"].describe())

تعداد انتخاب از هر امتیاز: {1: 7889, 2: 5422, 3: 7753, 4: 14970, 5: 43966} | مجموع: 80000
امتیاز 1: انتخاب 7889 از 79730 | کمترین vote انتخاب‌شده: 5 | هم‌vote در مرز: 2040 (انتخاب 165)
امتیاز 2: انتخاب 5422 از 54794 | کمترین vote انتخاب‌شده: 4 | هم‌vote در مرز: 1396 (انتخاب 1338)
امتیاز 3: انتخاب 7753 از 78356 | کمترین vote انتخاب‌شده: 3 | هم‌vote در مرز: 3417 (انتخاب 137)
امتیاز 4: انتخاب 14970 از 151281 | کمترین vote انتخاب‌شده: 3 | هم‌vote در مرز: 6041 (انتخاب 371)
امتیاز 5: انتخاب 43966 از 444315 | کمترین vote انتخاب‌شده: 4 | هم‌vote در مرز: 10320 (انتخاب 8952)

توزیع امتیازها بعد از انتخاب:
overall
1    0.0986
2    0.0678
3    0.0969
4    0.1871
5    0.5496
Name: proportion, dtype: float64

vote در نمونه‌ی انتخاب‌شده:
count    80000.000000
mean        14.386250
std         40.277225
min          3.000000
25%          5.000000
50%          7.000000
75%         12.000000
max       2846.000000
Name: vote, dtype: float64


<div dir="rtl">

### منطق نمونه‌گیری
- سهم هر امتیاز از ۸۰ هزار، مطابق توزیع داده‌ی اصلی محاسبه شد (توزیع کلاس‌ها بعد از نمونه‌گیری تغییر نکرد).
- داخل هر امتیاز، نقدهای با vote بیشتر اولویت دارند. این نقدها معمولا مفصل‌تر و پراطلاعات‌ترند.
</div>

In [ ]:
rest, test_df = train_test_split(sel, test_size=N_TEST, stratify=sel["label"], random_state=SEED)
train_df, val_df = train_test_split(rest, test_size=N_VAL, stratify=rest["label"], random_state=SEED)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

print("train / val / test:", train_df.shape, val_df.shape, test_df.shape)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, d["overall"].value_counts(normalize=True).sort_index().round(3).to_dict())

# ذخیره‌ی split ها (برای استفاده‌ی مدل‌های دیگر روی دقیقاً همین تقسیم)
os.makedirs(f"{SAVE_ROOT}/data", exist_ok=True)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(f"{SAVE_ROOT}/data/{name}.csv", index=False)

train / val / test: (56000, 4) (16000, 4) (8000, 4)
train {1: 0.099, 2: 0.068, 3: 0.097, 4: 0.187, 5: 0.55}
val {1: 0.099, 2: 0.068, 3: 0.097, 4: 0.187, 5: 0.55}
test {1: 0.099, 2: 0.068, 3: 0.097, 4: 0.187, 5: 0.55}


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(texts, labels):
    ds = Dataset.from_dict({"text": list(texts), "label": [int(x) for x in labels]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds = make_ds(train_df["full_text"], train_df["label"])
val_ds   = make_ds(val_df["full_text"],   val_df["label"])

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"micro_f1": f1_score(p.label_ids, preds, average="micro"),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/56000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

In [ ]:
_ta = inspect.signature(TrainingArguments.__init__).parameters
EVAL_KW   = "eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy"
WARMUP_KW = "warmup_ratio"  if "warmup_ratio"  in _ta else "warmup_steps"
TOK_KW    = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"

# وزن‌های DeBERTa به‌صورت fp16 لود می‌شن؛ .float() برای سازگاری با fp16 mixed-precision
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5).float()

args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    weight_decay=0.01,
    **{WARMUP_KW: 0.06},
    **{EVAL_KW: "epoch"},
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    save_total_limit=1,
    logging_steps=200,
    fp16=True,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
    **{TOK_KW: tok},
)

has_ckpt = len(glob.glob(f"{CKPT_DIR}/checkpoint-*")) > 0
print("ادامه از چک‌پوینت:", has_ckpt)
trainer.train(resume_from_checkpoint=True if has_ckpt else None)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

ادامه از چک‌پوینت: False


model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5250, training_loss=0.6163851648966472, metrics={'train_runtime': 2056.6536, 'train_samples_per_second': 81.686, 'train_steps_per_second': 2.553, 'total_flos': 1.1051160164352e+16, 'train_loss': 0.6163851648966472, 'epoch': 3.0})

<div dir="rtl">

### نکته درباره‌ی انتخاب checkpoint
- `micro_f1` در epoch اول (۰٫۷۴۱۹) و سوم (۰٫۷۴۱۶) تقریباً برابر است، ولی val loss از ۰٫۶۴۴ تا ۰٫۶۸۱ بالا رفته است (overconfidence).
- با `load_best_model_at_end=True` مدل epoch اول نگه داشته شده است. به همین دلیل نتایج بعدی با epoch اول مطابقت دارند.

</div>

In [ ]:
def predict_probs(texts, bs=128):
    """احتمال softmax هر ۵ کلاس؛ ترتیب خروجی همان ترتیب ورودی است."""
    m = trainer.model.eval()
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])               # مرتب‌سازی بر اساس طول (فقط برای سرعت)
    out = np.zeros((len(texts), 5), dtype=np.float32)
    for i in range(0, len(texts), bs):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(m.device)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            out[idx] = m(**enc).logits.float().cpu().numpy()
    return softmax(out, axis=1)

def evaluate(name, d):
    probs = predict_probs(d["full_text"])
    y, pred = d["label"].values, probs.argmax(1)
    micro = f1_score(y, pred, average="micro")
    macro = f1_score(y, pred, average="macro")
    ll    = log_loss(y, probs, labels=range(5))
    se    = 1.96 * np.sqrt(micro * (1 - micro) / len(y))
    print(f"\n===== {name} ({len(y)} نمونه) =====")
    print(f"micro-F1: {micro:.4f} (±{se:.3f}) | macro-F1: {macro:.4f} | log-loss: {ll:.4f}")
    print(classification_report(y, pred, target_names=["1", "2", "3", "4", "5"], zero_division=0))
    print("Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):")
    print(confusion_matrix(y, pred))
    return probs

probs_val  = evaluate("validation", val_df)
probs_test = evaluate("test", test_df)

trainer.save_model(f"{SAVE_ROOT}/model")
tok.save_pretrained(f"{SAVE_ROOT}/model")
np.save(f"{SAVE_ROOT}/val_probs.npy",  probs_val)
np.save(f"{SAVE_ROOT}/test_probs.npy", probs_test)
print("\nذخیره شد:", SAVE_ROOT)


===== validation (16000 نمونه) =====
micro-F1: 0.7418 (±0.007) | macro-F1: 0.5885 | log-loss: 0.6445
              precision    recall  f1-score   support

           1       0.68      0.80      0.74      1578
           2       0.47      0.23      0.31      1084
           3       0.48      0.62      0.54      1551
           4       0.64      0.37      0.47      2994
           5       0.84      0.94      0.89      8793

    accuracy                           0.74     16000
   macro avg       0.62      0.59      0.59     16000
weighted avg       0.73      0.74      0.72     16000

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[1270  115  122   16   55]
 [ 399  252  363   25   45]
 [ 128  140  959  205  119]
 [  46   20  470 1106 1352]
 [  30    4   98  380 8281]]

===== test (8000 نمونه) =====
micro-F1: 0.7369 (±0.010) | macro-F1: 0.5816 | log-loss: 0.6505
              precision    recall  f1-score   support

           1       0.65      0.77      0.70       789
           2     

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


ذخیره شد: /content/drive/MyDrive/NLP_Proj/deberta_80k


<div dir="rtl">

### ارزیابی روی نمونه‌ی «طبیعی»
۵۰۰۰ نقد تصادفی از داده‌ی اصلی که هیچ هم‌پوشانی‌ای با 80K ندارد (میانگین vote برابر ۰٫۳۶).<br> هدف بررسی این است که آیا انتخاب پر-vote باعث ضعیف‌شدن مدل روی نقدهای معمولی شده است یا نه. نتیجه: micro-F1 برابر ۰٫۷۴۶ (کمی بهتر از val).

</div>

In [ ]:
nat = pd.read_csv(CSV_PATH, usecols=["overall", "vote", "full_text"])
nat["overall"] = nat["overall"].astype(int)
nat["vote"]    = pd.to_numeric(nat["vote"], errors="coerce").fillna(0).astype(int)
nat = nat.dropna(subset=["full_text"]).drop_duplicates(subset=["full_text"])
nat = nat[~nat["full_text"].isin(set(sel["full_text"]))]            # هیچ هم‌پوشانی با ۸۰ هزار
nat = nat.sample(5000, random_state=SEED).reset_index(drop=True)
nat["label"] = nat["overall"] - 1

print("vote (طبیعی):", nat["vote"].describe()[["mean", "50%", "max"]].round(2).to_dict(),
      "| vote (train ۸۰K):", train_df["vote"].describe()[["mean", "50%", "max"]].round(2).to_dict())
print("میانگین تعداد کلمه  طبیعی:", round(nat["full_text"].str.split().str.len().mean(), 1),
      "| train:", round(train_df["full_text"].str.split().str.len().mean(), 1))
print("توزیع طبیعی:", nat["overall"].value_counts(normalize=True).sort_index().round(3).to_dict())

probs_nat = evaluate("natural (تصادفی، خارج از 80K)", nat)
np.save(f"{SAVE_ROOT}/natural_probs.npy", probs_nat)
nat[["overall", "label"]].to_csv(f"{SAVE_ROOT}/data/natural_5k_labels.csv", index=False)

vote (طبیعی): {'mean': 0.36, '50%': 0.0, 'max': 5.0} | vote (train ۸۰K): {'mean': 14.41, '50%': 7.0, 'max': 2846.0}
میانگین تعداد کلمه  طبیعی: 112.7 | train: 226.6
توزیع طبیعی: {1: 0.099, 2: 0.068, 3: 0.099, 4: 0.192, 5: 0.542}

===== natural (تصادفی، خارج از 80K) (5000 نمونه) =====
micro-F1: 0.7460 (±0.012) | macro-F1: 0.5945 | log-loss: 0.6147
              precision    recall  f1-score   support

           1       0.65      0.83      0.73       494
           2       0.42      0.22      0.29       342
           3       0.49      0.59      0.54       497
           4       0.69      0.42      0.52       958
           5       0.85      0.94      0.89      2709

    accuracy                           0.75      5000
   macro avg       0.62      0.60      0.59      5000
weighted avg       0.73      0.75      0.73      5000

Confusion matrix (سطر: واقعی، ستون: پیش‌بینی):
[[ 408   37   38    1   10]
 [ 147   76  106    8    5]
 [  45   59  295   56   42]
 [  14    7  132  405  400]
 [  